# Gradient Clipping & Stabilization

This notebook explores gradient clipping and normalization techniques for stabilizing neural network training.

**Learning Objectives:**
- Understand gradient explosion and vanishing problems
- Learn how gradient clipping works mathematically and geometrically
- Compare value clipping vs norm clipping
- Know when to use (and when NOT to use) gradient clipping
- Master practical implementation in PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import math
from collections import defaultdict

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

plt.rcParams['figure.figsize'] = (12, 6)

---
# Part 1: The Gradient Explosion Problem

## What is Gradient Explosion?

During backpropagation, gradients can grow exponentially large, especially in:
- **Recurrent networks** (RNNs, LSTMs) due to repeated matrix multiplications
- **Very deep networks** where gradients are multiplied across many layers
- **Networks with poor initialization** or high learning rates

When gradients explode:
1. Weight updates become massive: `w_new = w_old - lr * huge_gradient`
2. Weights jump to extreme values
3. Loss becomes NaN or infinity
4. Training fails catastrophically

In [ ]:
# Simulate gradient flow through layers
def simulate_gradient_propagation(num_layers, weight_multiplier):
    """Simulate gradient flow through layers with constant weight multiplier."""
    gradients = [1.0]  # Start with gradient of 1
    for layer in range(num_layers):
        gradients.append(gradients[-1] * weight_multiplier)
    return gradients

num_layers = 20
scenarios = {
    'Exploding (×1.2)': 1.2,
    'Stable (×1.0)': 1.0,
    'Vanishing (×0.8)': 0.8,
    'Severe Vanishing (×0.5)': 0.5
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for label, multiplier in scenarios.items():
    grads = simulate_gradient_propagation(num_layers, multiplier)
    ax1.plot(range(len(grads)), grads, marker='o', label=label)
    ax2.semilogy(range(len(grads)), grads, marker='o', label=label)

ax1.set_xlabel('Layer (from output to input)')
ax1.set_ylabel('Gradient Magnitude')
ax1.set_title('Gradient Flow (Linear Scale)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Layer (from output to input)')
ax2.set_ylabel('Gradient Magnitude (log scale)')
ax2.set_title('Gradient Flow (Log Scale)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nWith 20 layers:")
print(f"  Exploding: 1.2^20 = {1.2**20:.1f}x magnification")
print(f"  Vanishing: 0.8^20 = {0.8**20:.4f}x (gradient nearly gone!)")

---
# Part 2: Gradient Clipping - The Math

## Solution 1: Gradient Clipping by Value

Clip each gradient element independently:
```
g_clipped[i] = clip(g[i], -threshold, +threshold)
```

**PyTorch:** `torch.nn.utils.clip_grad_value_(parameters, clip_value)`

## Solution 2: Gradient Clipping by Norm (Better!)

Rescale the entire gradient vector if its norm exceeds a threshold:
```
g_norm = ||g||₂
if g_norm > threshold:
    g_clipped = g * (threshold / g_norm)
else:
    g_clipped = g
```

**PyTorch:** `torch.nn.utils.clip_grad_norm_(parameters, max_norm)`

**Key difference**: Norm clipping preserves gradient **direction**, value clipping does not!

In [ ]:
# Demonstrate the difference
gradient = torch.tensor([3.0, 4.0])  # Magnitude = 5.0
print(f"Original gradient: {gradient}")
print(f"Original magnitude (L2 norm): {gradient.norm().item():.2f}")

# Clipping by value
value_threshold = 2.0
clipped_by_value = torch.clamp(gradient, -value_threshold, value_threshold)
print(f"\nClipped by value (threshold={value_threshold}): {clipped_by_value}")
print(f"New magnitude: {clipped_by_value.norm().item():.2f}")

# Clipping by norm
norm_threshold = 2.0
grad_norm = gradient.norm()
if grad_norm > norm_threshold:
    clipped_by_norm = gradient * (norm_threshold / grad_norm)
else:
    clipped_by_norm = gradient
    
print(f"\nClipped by norm (threshold={norm_threshold}): {clipped_by_norm}")
print(f"New magnitude: {clipped_by_norm.norm().item():.2f}")

# Compare directions
original_direction = gradient / gradient.norm()
value_direction = clipped_by_value / clipped_by_value.norm()
norm_direction = clipped_by_norm / clipped_by_norm.norm()

print("\n--- Direction Comparison ---")
print(f"Original direction: {original_direction.numpy()}")
print(f"After value clipping: {value_direction.numpy()}")
print(f"After norm clipping: {norm_direction.numpy()}")
print("\n✓ Norm clipping PRESERVES direction, value clipping does NOT!")

---
# Part 3: Geometric Intuition

Think of gradients as arrows in parameter space:
- **Direction** tells us which way to move
- **Magnitude** tells us how far to move

Gradient clipping creates a "sphere of acceptable gradients" - any gradient outside gets projected back.

In [ ]:
# Visualize gradient clipping geometrically
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

threshold = 2.0
np.random.seed(42)
gradients = np.random.randn(15, 2) * 3

# Value clipping (square boundary)
ax1.set_xlim(-5, 5)
ax1.set_ylim(-5, 5)
square = plt.Rectangle((-threshold, -threshold), 2*threshold, 2*threshold,
                       fill=False, edgecolor='blue', linewidth=2, linestyle='--')
ax1.add_patch(square)

for g in gradients:
    g_clipped = np.clip(g, -threshold, threshold)
    ax1.arrow(0, 0, g[0], g[1], head_width=0.2, head_length=0.15,
             fc='red', ec='red', alpha=0.3, width=0.05)
    if np.linalg.norm(g) > threshold:
        ax1.arrow(0, 0, g_clipped[0], g_clipped[1], head_width=0.2, head_length=0.15,
                 fc='green', ec='green', width=0.05)

ax1.set_title('Value Clipping (Square Boundary)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Gradient[0]')
ax1.set_ylabel('Gradient[1]')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='k', linewidth=0.5)
ax1.axvline(x=0, color='k', linewidth=0.5)

# Norm clipping (circular boundary)
ax2.set_xlim(-5, 5)
ax2.set_ylim(-5, 5)
circle = Circle((0, 0), threshold, fill=False, edgecolor='blue', linewidth=2, linestyle='--')
ax2.add_patch(circle)

for g in gradients:
    g_norm = np.linalg.norm(g)
    if g_norm > threshold:
        g_clipped = g * (threshold / g_norm)
    else:
        g_clipped = g
    
    ax2.arrow(0, 0, g[0], g[1], head_width=0.2, head_length=0.15,
             fc='red', ec='red', alpha=0.3, width=0.05)
    if g_norm > threshold:
        ax2.arrow(0, 0, g_clipped[0], g_clipped[1], head_width=0.2, head_length=0.15,
                 fc='green', ec='green', width=0.05)

ax2.set_title('Norm Clipping (Circular Boundary)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Gradient[0]')
ax2.set_ylabel('Gradient[1]')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='k', linewidth=0.5)
ax2.axvline(x=0, color='k', linewidth=0.5)
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

print("Light red arrows: Original gradients")
print("Green arrows: Clipped gradients")
print("\nKey difference: Norm clipping preserves DIRECTION!")

---
# Part 4: Training Example

Let's train a deep network and see gradient clipping in action.

In [ ]:
class DeepNetwork(nn.Module):
    """A deep network to demonstrate gradient issues."""
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(input_size, hidden_size)])
        for _ in range(num_layers - 2):
            self.layers.append(nn.Linear(hidden_size, hidden_size))
        self.layers.append(nn.Linear(hidden_size, output_size))
    
    def forward(self, x):
        for layer in self.layers[:-1]:
            x = torch.tanh(layer(x))
        return self.layers[-1](x)

def compute_gradient_norm(model):
    """Compute total gradient norm across all parameters."""
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item() ** 2
    return math.sqrt(total_norm)

# Create synthetic dataset
X = torch.randn(800, 10)
y = (X.sum(dim=1) > 0).long()
dataset = TensorDataset(X, y)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

print("Created deep network and synthetic dataset")

In [ ]:
def train_model(model, train_loader, num_epochs, learning_rate, clip_value=None):
    """Train model and track gradient statistics."""
    model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    
    history = {'loss': [], 'grad_norm': []}
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        epoch_grad_norms = []
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            output = model(batch_x)
            loss = F.cross_entropy(output, batch_y)
            loss.backward()
            
            # Compute gradient norm BEFORE clipping
            grad_norm = compute_gradient_norm(model)
            epoch_grad_norms.append(grad_norm)
            
            # Gradient clipping (if specified)
            if clip_value is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)
            
            optimizer.step()
            epoch_loss += loss.item()
        
        history['loss'].append(epoch_loss / len(train_loader))
        history['grad_norm'].append(np.mean(epoch_grad_norms))
        
        if epoch % 10 == 0 or epoch == num_epochs - 1:
            print(f"Epoch {epoch:3d} | Loss: {history['loss'][-1]:.4f} | Grad Norm: {history['grad_norm'][-1]:.2f}")
    
    return history

In [ ]:
# Train WITHOUT clipping
print("Training WITHOUT gradient clipping...\n")
torch.manual_seed(42)
model_no_clip = DeepNetwork(10, 64, 2, num_layers=10).to(device)
history_no_clip = train_model(model_no_clip, train_loader, num_epochs=50, 
                               learning_rate=0.1, clip_value=None)

In [ ]:
# Train WITH clipping
print("\nTraining WITH gradient clipping (max_norm=1.0)...\n")
torch.manual_seed(42)
model_with_clip = DeepNetwork(10, 64, 2, num_layers=10).to(device)
history_with_clip = train_model(model_with_clip, train_loader, num_epochs=50,
                                learning_rate=0.1, clip_value=1.0)

In [ ]:
# Compare results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_no_clip['loss'], 'r-', linewidth=2, label='No Clipping', alpha=0.7)
ax1.plot(history_with_clip['loss'], 'g-', linewidth=2, label='With Clipping', alpha=0.7)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss Comparison', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.semilogy(history_no_clip['grad_norm'], 'r-', linewidth=2, label='No Clipping', alpha=0.7)
ax2.semilogy(history_with_clip['grad_norm'], 'g-', linewidth=2, label='With Clipping', alpha=0.7)
ax2.axhline(y=1.0, color='blue', linestyle='--', linewidth=2, label='Clipping Threshold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Gradient Norm (log scale)')
ax2.set_title('Gradient Norms Comparison', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("COMPARISON SUMMARY")
print("="*60)
print(f"\nWithout clipping: Max grad norm = {max(history_no_clip['grad_norm']):.2f}")
print(f"With clipping:    Max grad norm = {max(history_with_clip['grad_norm']):.2f}")

---
# Part 5: Choosing the Right Threshold

**Guidelines:**
1. Monitor gradient norms during initial training (without clipping)
2. Choose threshold around 95th-99th percentile
3. Aim to clip only outliers (5-20% of gradients)

**Common values:**
- RNNs: 1.0 - 5.0
- Transformers: 1.0
- Deep CNNs: 5.0 - 10.0

In [ ]:
# Test different thresholds
thresholds = [0.5, 1.0, 2.0, 5.0, 10.0]
results = {}

print("Testing different clipping thresholds...\n")
for threshold in thresholds:
    torch.manual_seed(42)
    model = DeepNetwork(10, 64, 2, num_layers=10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    
    grad_norms = []
    for epoch in range(20):
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(batch_x), batch_y)
            loss.backward()
            grad_norms.append(compute_gradient_norm(model))
            torch.nn.utils.clip_grad_norm_(model.parameters(), threshold)
            optimizer.step()
    
    clipped_pct = 100 * sum(1 for g in grad_norms if g > threshold) / len(grad_norms)
    results[threshold] = {'grad_norms': grad_norms, 'clipped_pct': clipped_pct}
    print(f"Threshold {threshold:4.1f}: {clipped_pct:.1f}% of gradients clipped")

print("\n💡 Recommendation: Choose threshold where 5-20% of gradients are clipped")

---
# Part 6: When to Use Gradient Clipping

## ✅ USE gradient clipping when:
1. **Training RNNs/LSTMs/GRUs** - Almost always beneficial
2. **Training very deep networks** (10+ layers without residual connections)
3. **Training Transformers** - Common practice
4. **Using high learning rates**
5. **Fine-tuning pre-trained models**
6. **Observing NaN/Inf losses or gradient spikes**

## ❌ DON'T use gradient clipping when:
1. **Training shallow networks** (3-5 layers)
2. **Networks with batch normalization** (already stabilizes gradients)
3. **Using Adam with default LR** (usually stable)
4. **Training is already stable** (don't add unnecessary complexity)
5. **Better alternatives exist** (fix initialization, architecture, or LR first)

---
# Part 7: PyTorch Implementation Patterns

In [ ]:
# Pattern 1: Basic gradient clipping
print("Pattern 1: Basic Gradient Clipping")
print("="*50)
print("""
loss.backward()  # Compute gradients
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Clip
optimizer.step()  # Update

✓ Always: backward() → clip() → step()
""")

# Pattern 2: With monitoring
print("\nPattern 2: Gradient Clipping with Monitoring")
print("="*50)
print("""
loss.backward()
total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(f"Gradient norm (before clipping): {total_norm:.2f}")
optimizer.step()

✓ clip_grad_norm_() returns the norm BEFORE clipping
""")

# Pattern 3: Value vs Norm clipping
print("\nPattern 3: Value vs Norm Clipping")
print("="*50)
print("""
# Norm clipping (RECOMMENDED)
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

# Value clipping (clips each element independently)
torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=1.0)

✓ Use norm clipping unless you have a specific reason not to!
""")

---
# Summary: Key Takeaways

## Core Concepts
1. **Gradient Explosion**: Gradients grow exponentially through deep networks
2. **Gradient Clipping**: Limits gradient magnitude while preserving direction (for norm clipping)
3. **Norm vs Value**: Norm clipping is preferred (preserves direction)

## Practical Guidelines
| Scenario | Recommended max_norm |
|----------|---------------------|
| RNNs/LSTMs | 1.0 - 5.0 |
| Transformers | 1.0 - 5.0 |
| Deep CNNs | 1.0 |
| General | Start with 1.0 |

## Decision Tree
```
Training unstable (NaN, spikes)?
├─ YES → Add gradient clipping
└─ NO
   ├─ Training RNN/LSTM? → Add clipping (standard practice)
   ├─ Very deep network? → Consider clipping
   └─ Already stable? → Don't add unnecessary complexity
```

## Common Mistakes
1. ❌ Clipping in wrong order (must be: backward → clip → step)
2. ❌ Threshold too low (clipping >50% of gradients = too aggressive)
3. ❌ Using value clipping when norm clipping is better
4. ❌ Not monitoring gradient norms

**Remember**: Gradient clipping is a symptom treatment. If training is consistently unstable, also investigate root causes (initialization, architecture, learning rate).